# Logistic Regression Assignment (Fixed Version)

Handles common errors automatically.

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

import statsmodels.api as sm


In [24]:
# Load dataset (update filename if needed)
try:
    df = pd.read_csv("sample (1).csv")
except:
    df = pd.read_csv("sample.csv")

print("Loaded dataset successfully")
df.head()


Loaded dataset successfully


,female,read,write,math,hon,femalexmath
0,0,57,52,41,0,0
1,1,68,59,53,0,53
2,0,44,33,54,0,0
3,0,63,44,47,0,0
4,0,47,52,57,0,0


In [25]:
print(df.info())
print(df.describe())
print("Columns:", df.columns.tolist())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   female       200 non-null    int64
 1   read         200 non-null    int64
 2   write        200 non-null    int64
 3   math         200 non-null    int64
 4   hon          200 non-null    int64
 5   femalexmath  200 non-null    int64
dtypes: int64(6)
memory usage: 9.5 KB
None
          female        read       write        math         hon  femalexmath
count  200.00000  200.000000  200.000000  200.000000  200.000000   200.000000
mean     0.54500   52.230000   52.775000   52.645000    0.245000    28.555000
std      0.49922   10.252937    9.478586    9.368448    0.431166    27.011201
min      0.00000   28.000000   31.000000   33.000000    0.000000     0.000000
25%      0.00000   44.000000   45.750000   45.000000    0.000000     0.000000
50%      1.00000   50.000000   54.000000   52.000000    0.000000 

In [26]:
# Auto-detect target column
if 'hon' in df.columns:
    target_column = 'hon'
else:
    target_column = df.columns[-1]  # fallback

print("Using target column:", target_column)

X = df.drop(columns=[target_column])
y = df[target_column]

# Convert categorical variables safely
X = pd.get_dummies(X, drop_first=True)

# Ensure numeric
X = X.apply(pd.to_numeric, errors='coerce')
X = X.fillna(0)


Using target column: hon


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [28]:
model = LogisticRegression(max_iter=2000)

try:
    model.fit(X_train, y_train)
    print("Model trained successfully")
except Exception as e:
    print("Error during training:", e)


Model trained successfully


In [29]:
y_pred = model.predict(X_test)

print("CONFUSION MATRIX:")
print(confusion_matrix(y_test, y_pred))

print("\nCLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred))


CONFUSION MATRIX:
[[33  0]
 [ 0  7]]

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        33
           1       1.00      1.00      1.00         7

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



In [30]:
print("Intercept:", model.intercept_[0])

for name, coef in zip(X.columns, model.coef_[0]):
    print(f"{name}: {coef}")


Intercept: -140.83521419417855
female: 0.01246064420800183
read: -0.032486158387504856
write: 2.357022816016775
math: 0.001007518008404639
femalexmath: -0.0017560157397765882


In [31]:
def interpret_logistic_regression(model, feature_names):
    intercept = model.intercept_[0]
    coefs = model.coef_[0]

    print("Coefficients:")
    print(f"(Intercept): {intercept:.4f}")

    for name, coef in zip(feature_names, coefs):
        print(f"{name}: {coef:.4f}")

    print("\n--- Interpretation ---\n")

    odds = np.exp(intercept)
    print(f"Baseline log-odds: {intercept:.4f}")
    print(f"Baseline odds: {odds:.4f}")

    for name, coef in zip(feature_names, coefs):
        odds_ratio = np.exp(coef)
        percent_change = (odds_ratio - 1) * 100

        print(f"\n{name}:")
        print(f"Odds ratio = {odds_ratio:.4f}")

        if percent_change > 0:
            print(f"Increases odds by {percent_change:.2f}%")
        else:
            print(f"Decreases odds by {abs(percent_change):.2f}%")


In [32]:
interpret_logistic_regression(model, X.columns)


Coefficients:
(Intercept): -140.8352
female: 0.0125
read: -0.0325
write: 2.3570
math: 0.0010
femalexmath: -0.0018

--- Interpretation ---

Baseline log-odds: -140.8352
Baseline odds: 0.0000

female:
Odds ratio = 1.0125
Increases odds by 1.25%

read:
Odds ratio = 0.9680
Decreases odds by 3.20%

write:
Odds ratio = 10.5595
Increases odds by 955.95%

math:
Odds ratio = 1.0010
Increases odds by 0.10%

femalexmath:
Odds ratio = 0.9982
Decreases odds by 0.18%


## Interpretation Notes
- Intercept = log-odds of reference group
- exp(coefficient) = odds ratio
- >1 means more likely, <1 means less likely


In [33]:
# Statsmodels (safe version)
try:
    X_sm = sm.add_constant(X)
    model_sm = sm.Logit(y, X_sm)
    result = model_sm.fit(disp=0)
    print(result.summary())
except Exception as e:
    print("Statsmodels error (often due to perfect separation):", e)


Statsmodels error (often due to perfect separation): Singular matrix


C:\Users\lolim\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
C:\Users\lolim\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


## Final Interpretation Paragraph

The intercept represents the log-odds of the reference group (males) being in the honors class. 
The coefficient for female represents the change in log-odds compared to males. 
The odds ratio shows how much more likely females are to be in honors compared to males. 
If the odds ratio is greater than 1, females are more likely to be in honors.
